# Qwen3-0.6B × FlexAttention-backed AttentionBackend (educational)

**자매 프로젝트**: `../flashinfer/` (FlashInfer wrapper), `../flashattn/` (FlashAttention-2), `../ptr/vllm_unified/` (직접 작성한 Triton 커널).

이 노트북은 같은 4단 구조(Metadata/Builder/Impl/Backend)를 유지하며, 다른 것은 두 곳뿐:

- **cell 7**: CSR 변환 데모 자리 → **`physical_to_logical` 역표 데모**
- **cell 10**: LLM 로드 시 플러그인이 `CUSTOM` 슬롯에 `MyFlexAttnBackend` 를 올림

**핵심 교육 포인트 — BlockMask 브리지**:

> BlockMask 는 vLLM 의 물리 paged KV layout 과 순수-Python `mask_mod` 사이의 브리지다.
> `mask_mod` 는 `physical_to_logical` 역매핑을 통해 논리 토큰 인덱스를 받으므로
> 커널 자체는 paging 에 완전히 무관하다.

**이 프로젝트가 flashinfer 와 다른 점**:
- flashinfer 는 prefill/decode wrapper 가 분리 → split-dispatch 강제
- flex_attention + BlockMask 는 단일 launch 로 prefill/decode/chunked 통합
- 외부 라이브러리 불필요 (torch >= 2.5 만)

## 준비물 & 설치

- CUDA GPU 필수 (Hopper/Ampere/Blackwell). flex_attention 은 SM 8.0+ 권장.
- Python >= 3.10
- vLLM **0.19.1 정확히** 권장 (내부 API drift)
- torch >= 2.5 (flex_attention + BlockMask 안정화 버전)

```bash
# 노트북 디렉토리에서
pip install -e .
# 다른 CUSTOM backend 와 같은 venv 에 함께 설치 금지 — CUSTOM 슬롯 충돌.
```

`pyproject.toml` 의 entry point (`vllm.general_plugins = flex_attn_attention_backend:register`) 가
vLLM 프로세스에서 자동으로 `register()` 를 호출해 준다.

절차:
1. cell 3 — 환경 확인 (cuda + vllm + torch)
2. cell 8 — `physical_to_logical` 역표 데모
3. cell 11 — LLM 로드 (plugin 이 이미 CUSTOM 슬롯 점유)
4. cell 12~14 — generate + 실행 증거 확인

In [ ]:
import torch, vllm

print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm:', vllm.__version__)
print('torch:', torch.__version__)

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
    '다른 버전은 AttentionBackend 내부 API 가 다를 수 있음.'
)

# flex_attention 가용 여부 확인
from torch.nn.attention.flex_attention import flex_attention, create_block_mask, BlockMask
print('flex_attention: OK (torch', torch.__version__, ')')

## Qwen3-0.6B 구조 요약

| 항목 | 값 | flex_attention 비고 |
|---|---|---|
| hidden_size | 1024 | — |
| Q heads | 16 | — |
| KV heads | 8 (GQA 2:1) | `enable_gqa=True` |
| head_dim | 128 | flex_attention 은 head_dim 제한 없음 |
| layers | 28 | BlockMask 는 28회가 아닌 **1회** 구축됨 |

vLLM `block_size` 기본값 16 — BlockMask 의 BLOCK_SIZE 와 일치해야 함.

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden:', cfg.hidden_size)
print('Q heads:', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers:', cfg.num_hidden_layers)

## FlexAttention 백엔드의 역할 — BlockMask 브리지

flashinfer 와 달리, flex_attention 은 prefill/decode 구분 없이 **단일 launch**:

```
flashinfer:    plan(prefill) + run(prefill)   # q_len > 1
               plan(decode)  + run(decode)    # q_len == 1
               → 최대 2 launch

flex_attention: flex_attention(q, k_flat, v_flat, block_mask=...)  → 1 launch
```

대신 mask_mod 가 paged KV 의 물리 주소를 논리 주소로 변환해야 한다:

```
kv_idx (물리적, 0..total_cache_tokens)
  ↓ physical_to_logical 역표
logical_kv (논리적, 0..seq_len)
  ↓ causal mask
logical_kv <= logical_q_idx
```

**MetadataBuilder.build()** 가 이 `physical_to_logical` 표를 구축하고
`create_block_mask(mask_mod, ...)` 로 `BlockMask` 를 만든다.
28개 레이어가 이것을 공유한다.

## physical_to_logical 역표 데모

이 셀이 `qwen3_flashinfer_attention.ipynb` 의 **CSR 변환 데모** 자리에 해당한다.

tiny synthetic block_table 로 역표가 어떻게 구축되는지 확인한다.

In [ ]:
import torch
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from flex_attn_attention_backend import _build_physical_to_logical

# 3개 시퀀스, block_size=16, num_gpu_blocks=8
# seq 0: 길이 44 → 3블록, 물리 블록 [3, 7, 5]
# seq 1: 길이 21 → 2블록, 물리 블록 [1, 4]
# seq 2: 길이 16 → 1블록, 물리 블록 [6]
block_size = 16
num_gpu_blocks = 8

block_table = torch.tensor([
    [3, 7, 5, 0, 0],   # seq 0
    [1, 4, 0, 0, 0],   # seq 1
    [6, 0, 0, 0, 0],   # seq 2
], dtype=torch.int32, device='cuda')
seq_lens = torch.tensor([44, 21, 16], dtype=torch.int32, device='cuda')

p2l = _build_physical_to_logical(block_table, seq_lens, block_size, num_gpu_blocks)
print('physical_to_logical (shape:', p2l.shape, '):')
for i in range(3):
    print(f'  req {i}:', p2l[i].tolist())

print()
print('예시: kv_idx=80 이 req 0 에서 어떤 논리 위치?')
kv_idx = 80
phys_block = kv_idx // block_size
offset = kv_idx % block_size
log_block = p2l[0, phys_block].item()
print(f'  phys_block = {kv_idx} // {block_size} = {phys_block}')
print(f'  log_block  = physical_to_logical[0, {phys_block}] = {log_block}')
print(f'  logical_kv = {log_block} * {block_size} + {offset} = {log_block * block_size + offset}')

# 수동 검증
assert p2l[0, 3].item() == 0, 'phys 3 → logical 0 (첫 블록)'
assert p2l[0, 7].item() == 1, 'phys 7 → logical 1 (두 번째 블록)'
assert p2l[0, 5].item() == 2, 'phys 5 → logical 2 (세 번째 블록)'
assert p2l[1, 1].item() == 0 and p2l[1, 4].item() == 1, 'seq 1 확인'
assert p2l[2, 6].item() == 0, 'seq 2 확인'
assert p2l[0, 0].item() == -1, 'phys 0 (null block) 는 항상 -1'
print()
print('physical_to_logical 역표 PASSED')
print('이 역표가 매 forward builder.build() 에서 구축되어 mask_mod 클로저가 캡처한다.')

## Plugin + BlockMask 연결

`pyproject.toml`:
```toml
[project.entry-points."vllm.general_plugins"]
my_flexattn_backend = "flex_attn_attention_backend:register"
```

**Backend 의 단일 launch 로직** (`MyFlexAttnImpl.forward`):
```python
# KV 쓰기
triton_reshape_and_cache_flash(key[:N], value[:N], key_cache, value_cache, slot_mapping, ...)

# 물리 → flat 4D 변환
key_flat = key_cache.view(-1, Hkv, D)[None].permute(0, 2, 1, 3)  # (1, Hkv, total, D)
q4d      = query[None, :N].permute(0, 2, 1, 3)                    # (1, Hq,  N,     D)

# 단일 flex_attention 호출 — BlockMask 가 sparsity + causality 를 통합
out = flex_attention(q4d, key_flat, val_flat,
                     block_mask=attn_metadata.block_mask,
                     scale=self.scale, enable_gqa=(Hkv != Hq))
```

flashinfer 의 `plan(prefill) → run(prefill)` / `plan(decode) → run(decode)` 두 분기 대비
코드가 훨씬 단순하다.

In [ ]:
from importlib.metadata import entry_points

eps = list(entry_points(group='vllm.general_plugins'))
mine = [e for e in eps if e.name == 'my_flexattn_backend']
assert mine, '`my_flexattn_backend` entry point 가 보이지 않는다. pip install -e . 확인.'
print('plugin registered:', mine[0].name, '->', mine[0].value)

In [ ]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

# max_num_batched_tokens=64 로 chunked prefill trigger.
# flashinfer 와 달리 chunked 도 동일한 단일 launch 로 처리된다.
llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.CUSTOM,
    enforce_eager=True,
    max_num_seqs=4,
    max_model_len=2048,
    max_num_batched_tokens=64,
)

In [ ]:
long_prompt = (
    'In the long history of artificial intelligence research, from the early '
    'symbolic AI of the 1950s through the neural network revival of the 1980s, '
    'the deep learning breakthroughs of the 2010s, and the transformer-based '
    'large language models of the 2020s, one theme has remained constant: '
    'the answer is'
)

prompts = [
    'The capital of France is',
    long_prompt,
    'Shakespeare wrote the play',
    'Python was created by',
]
out = llm.generate(prompts, SamplingParams(temperature=0, max_tokens=16))
for i, o in enumerate(out):
    print(f'[{i}] (prompt {len(prompts[i])} chars) {o.outputs[0].text[:80]}')

## 검증 — 단일 launch 로 prefill+decode+chunked 가 모두 처리되어야 한다

엔진 코어 stderr 에 다음 같은 로그가 찍혀있어야:

```
MyFlexAttnImpl.forward fired num_seqs=N max_q_len=Q tokens=T n_chunked=C
```

**관찰 포인트**:
- `n_chunked > 0` 인 줄 → chunked prefill 이 단일 launch 에 포함됨
- flashinfer 의 `prefill=P decode=D` 두 카운터 대신 통합된 `n_chunked=C` 하나
- decode 중인 세션도 같은 launch 에서 처리됨

**flashinfer 와의 관찰 포인트 대조**:

| | flashinfer | flexattn (이 프로젝트) |
|---|---|---|
| launch 수 | 최대 2 (prefill + decode 분리) | **항상 1** |
| chunked 위치 | prefill_calls 에 합산 | n_chunked 로 별도 카운트 |
| prefill+decode 공존 증거 | `prefill=P>0 AND decode=D>0` | 같은 launch 안에서 처리 |

**BlockMask 구축 타이밍**: `build()` 에서 1회 → 28개 레이어 공유.
`Impl.forward` 에서 28번 구축하지 않는다.

In [ ]:
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.CUSTOM.get_path()
print('CUSTOM slot ->', path)
assert 'MyFlexAttnBackend' in path
print('OK — plugin 자동 등록 확인')
print()
print('FlexAttention 실행 증거 = 위 cell 출력 위의 `MyFlexAttnImpl.forward fired` 로그')
print('→ n_chunked > 0 인 줄이 있으면 chunked prefill 이 단일 launch 로 처리된 것')

## 이 프로젝트의 위치 & 마무리

```
vllm_attn/
  ├─ ptr/vllm_unified/   Triton 커널 1개, prefill+decode+chunked 통합
  ├─ flashinfer/         FlashInfer wrapper, split-dispatch (2 launch)
  ├─ flashattn/          FlashAttention-2
  └─ flexattn/  (← 여기) PyTorch flex_attention, BlockMask 브리지 (1 launch)
```

**backend.py diff 로 보이는 것**:

1. **KV cache layout 차이**: flashinfer = `(B, 2, bs, H, D)` / unbind(1), flexattn = `(2, B, bs, H, D)` / unbind(0).
   같은 `triton_reshape_and_cache_flash` 를 호출하지만 레이아웃이 다르다.

2. **Metadata 변환이 backend 특수**: flashinfer 는 CSR triple, flexattn 은 physical_to_logical 역표.
   이것이 라이브러리 교체의 실제 비용이다.

3. **단일 launch 의 비용**: flex_attention 은 mask_mod 평가를 위해 Python 수준 sparsity 그리드를
   `create_block_mask` 시점에 한 번 실행한다. torch.compile 을 켜면 이 비용이 줄어든다.

**단순화된 것 (이 프로젝트)**:
- `cudagraph_support = NEVER`
- `torch.compile(flex_attention, fullgraph=True)` 활성 — eager 호출은 sdpa_dense math fallback → OOM 이므로 compile 필수 (stock 과 동일)
- `create_block_mask` 를 eager 로 호출 (stock 은 compiled 버전 사용)
- Sliding window / alibi / soft cap / encoder-only 미지원